# Lab Assignment 7
## RNN Applications: from SimpleRNN to LSTM, and from Classification to Tagging


---

### Objective

The tutorial ended with a problem rather than a solution. A **SimpleRNN** classified sentiment at
about 64%, and its accuracy **collapsed on long sentences** — 70.5% on short ones, 58.4% on long
ones. The lecture blames the **vanishing gradient**, and claims an **LSTM** fixes it.

This assignment does two things.

**Part A — Task 1.** Test that claim properly. Same data, same task, same settings; you change
**only** the recurrent layer. If the lecture is right, the LSTM should help — and it should help
**most on the long sentences**.

**Part B — Tasks 2 to 5.** Take the LSTM to a different kind of problem: **POS tagging**, where
every word gets its own label instead of one label for the whole sentence. Then make it
**bidirectional** and measure what that buys you.

You tagged parts of speech once before, in **Lab 3, using a Hidden Markov Model**. This time the
model learns the patterns from data instead of from explicit transition and emission counts.

### How this differs from the tutorial

| | Tutorial demo | This assignment |
|---|---|---|
| Application | sentiment classification | sentiment (Task 1), then **POS tagging** (Tasks 2–5) |
| Shape | many-to-one | many-to-one, then **many-to-many** |
| Recurrent cell | `SimpleRNN` | **`LSTM`**, then **`Bidirectional(LSTM)`** |
| Output | `Dense(1)` on the final state | `Dense(1)`, then `TimeDistributed(Dense(n_tags))` |
| Data | movie review sentences | the same reviews, then the **Penn Treebank** |

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Does the LSTM actually fix it? | **4** |
| **Task 2** | Data preparation for tagging | **4** |
| **Task 3** | Building and training the LSTM tagger | **4** |
| **Task 4** | Evaluating the tagger correctly | **4** |
| **Task 5** | Bidirectional LSTM and comparison | **4** |
| | **Total** | **20** |

---

### Before you start

- A GPU is **not** required. Everything here runs in a few minutes on CPU.
- **Task 1 needs the file `mini_sentiment.tsv`**, the same one the tutorial used. You must have it
  in the same folder as this notebook before you start.
  **In Colab:** click the folder icon in the left sidebar and drag the file in, or run
  ```python
  from google.colab import files
  files.upload()
  ```
  Do **not** rebuild this file from NLTK yourself. The baseline numbers you are comparing against
  come from this exact file in this exact order, so a regenerated copy would make the comparison
  invalid.
- You may reuse the tutorial's Task 1 pipeline directly — the marks are for the **measurement and
  the explanation**, not for retyping the data loading.
- **Tasks 2–5** use the Penn Treebank from NLTK.
- Use the **universal tagset** (12 tags), not the full Penn tagset (45 tags). With only 3,914
  sentences the rarer Penn tags have far too few examples.

**You are expected to write the code yourself.** Each task states what to build, the names to use,
and the exact output it must print — but not how to write it.

---
## Setup

These are the only values given to you.

In [ ]:
import numpy as np
import nltk
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Bidirectional, Dense, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('treebank', quiet=True)
nltk.download('universal_tagset', quiet=True)
from nltk.corpus import treebank

keras.utils.set_random_seed(42)

# --- Task 1 (sentiment), same values as the tutorial ---
MAX_WORDS = 5000
SENT_LEN  = 40

# --- Tasks 2-5 (POS tagging) ---
MAX_LEN   = 50      # pad/truncate every sentence to this many words
EMBED_DIM = 64
RNN_UNITS = 64
N_TRAIN   = 3000    # first 3000 sentences for training, the rest for testing

tagged_sentences = treebank.tagged_sents(tagset='universal')
print(f"{len(tagged_sentences)} tagged sentences")
print("example:", tagged_sentences[0][:6])

---
# Task 1: Does the LSTM actually fix it? &nbsp;&nbsp;*(4 marks)*

The tutorial's SimpleRNN scored this on the sentiment test set:

| Group | n | SimpleRNN |
|---|---|---|
| SHORT (≤ 15 words) | 808 | **0.705** |
| ALL | 2662 | **0.639** |
| LONG (≥ 28 words) | 680 | **0.584** |

Rebuild that classifier **changing only the recurrent layer** — `SimpleRNN(32)` becomes
`LSTM(32)`. Everything else stays identical: same file, same `MAX_WORDS`, same `SENT_LEN`, same
8000/2662 split, same epochs, same batch size. If you change anything else the comparison is
meaningless.

Then score it the same three ways and put the numbers side by side.

### What your code must create

| Name | Is |
|---|---|
| `X_sent`, `y_sent` | the padded sentiment data, exactly as in the tutorial |
| `raw_len` | each test sentence's true word count, **before** padding |
| `model_lstm_sent` | the same architecture with `LSTM(32)` in place of `SimpleRNN(32)` |

### Output your code must print

```
GROUP                     n    SimpleRNN      LSTM     change
------------------------------------------------------------
SHORT (<= 15 words)     808        0.705     0.___     +_.___
ALL                    2662        0.639     0.___     +_.___
LONG  (>= 28 words)     680        0.584     0.___     +_.___
```

### Then answer

**Q1.** Where is the LSTM's improvement largest — on short sentences or long ones? Does that match
what the lecture predicts about vanishing gradients? Two or three sentences.

### Hints

- Read `mini_sentiment.tsv` exactly as the tutorial does: one header line, then `pos`/`neg` and the
  sentence separated by a tab.
- The split is the first 8000 sentences for training, the remainder for testing.
- `raw_len` must come from the token lists, **not** from the padded array — after padding every
  sentence has the same length.
- Call `keras.utils.set_random_seed(42)` before building the model.

In [ ]:
# ---------------- TASK 1 ----------------

# TODO 1: load mini_sentiment.tsv, build the vocabulary, encode and pad -> X_sent, y_sent
#         (same procedure as the tutorial)

# TODO 2: compute raw_len for the TEST sentences, from the token lists before padding

# TODO 3: build the same model as the tutorial but with LSTM(32) instead of SimpleRNN(32),
#         and train it with identical settings

# TODO 4: score it three ways -- SHORT (<=15 words), ALL, LONG (>=28 words)

# TODO 5: print the comparison table against the SimpleRNN numbers given above

---
# Task 2: Data preparation for tagging &nbsp;&nbsp;*(4 marks)*

Now switch tasks. Turn the tagged sentences into the rectangular integer arrays an RNN needs.

Each sentence is a list of `(word, tag)` pairs, so you need **two** parallel sequences: word ids
and tag ids, aligned position by position.

### What your code must create

| Name | Holds |
|---|---|
| `words`, `tags` | the sorted vocabularies (lowercase the words) |
| `word_index`, `tag_index` | word → id and tag → id dictionaries |
| `X`, `y` | padded word-id and tag-id arrays, shape `(n_sentences, MAX_LEN)` |
| `X_train`, `y_train`, `X_test`, `y_test` | split at `N_TRAIN` |

### Output your code must print

```
Vocabulary size: ...
Number of tags: ...
Tags: ['.', 'ADJ', 'ADP', ...]
X shape: (3914, 50)   y shape: (3914, 50)
Training sentences: 3000   Test sentences: 914
```
Then print the first 8 words and tags of one sentence, in both word form and id form, so it is
visible that the two sequences line up.

### Hints

- Reserve **id 0 for padding** in both vocabularies, and **id 1 for unknown words**. Real word ids
  start at 2, real tag ids at 1. Nothing may be given id 0 except padding.
- Lowercase words when building the vocabulary, and lowercase again when looking them up.
- Use `padding='post'` here. The tutorial padded at the *front*, which is fine when only the last
  hidden state matters — but for tagging, padding at the **end** keeps word `i` and tag `i` at the
  same index, which is what makes masking work.
- Pad `X` and `y` **the same way**.
- Check what shape `sparse_categorical_crossentropy` expects for the targets.

In [ ]:
# ---------------- TASK 2 ----------------

# TODO 6: build the word and tag vocabularies (`words`, `tags`), lowercasing the words

# TODO 7: build `word_index` and `tag_index`, leaving id 0 free for padding
#         (and id 1 for unknown words in word_index)

# TODO 8: convert every sentence into a list of word ids and a list of tag ids

# TODO 9: pad both to MAX_LEN with padding='post', giving `X` and `y`

# TODO 10: split into X_train, y_train, X_test, y_test at N_TRAIN

# TODO 11: print the statistics above, plus one example sentence in word form
#          and id form so the alignment is visible

---
# Task 3: Building and training the LSTM tagger &nbsp;&nbsp;*(4 marks)*

```
   word ids  ->  Embedding  ->  LSTM(return_sequences=True)  ->  TimeDistributed(Dense(softmax))
   (n, 50)       (n, 50, 64)         (n, 50, 64)                        (n, 50, n_tags+1)
```

Compare this with the tutorial's model. Two things changed, and both are essential:

1. **`return_sequences=True`** — the LSTM must output a hidden state at **every** time step, not
   just the last. Without it you get one vector for the whole sentence and cannot tag words.
2. **`TimeDistributed(Dense(...))`** — applies the same Dense classifier independently at every
   time step. One shared classifier, applied 50 times.

### What your code must create

| Name | Is |
|---|---|
| `model_lstm` | the compiled tagger |
| `history_lstm` | the return value of `model_lstm.fit(...)` |

### Output your code must print

- `model_lstm.summary()` — showing **real output shapes and parameter counts**, not `?` and `0`
- the per-epoch table, ending with the final `loss`, `accuracy`, `val_loss`, `val_accuracy`

A correct summary looks like this (total parameters **762,765**):

```
 embedding (Embedding)        (None, 50, 64)        728,896
 lstm (LSTM)                  (None, 50, 64)         33,024
 time_distributed             (None, 50, 13)            845
```

Read the middle column: the LSTM keeps all 50 positions, and the output has 13 values per position
— one score per tag. That is what `return_sequences=True` and `TimeDistributed` buy you.

### Hints

- **Start your `Sequential` with `Input(shape=(MAX_LEN,))`.** Without it Keras does not build the
  model until training begins, so `model.summary()` prints `?` for every output shape and
  `0 (unbuilt)` for every parameter count — useless for your submission. The same applies to your
  Task 1 model, with `SENT_LEN` instead.
- Use `Embedding(..., mask_zero=True)` so later layers ignore padded positions and the model is not
  trained to predict a tag for padding. Note what this does to the accuracy Keras reports — Task 4
  asks about it.
- The output layer needs one unit per tag id **including the reserved 0**, so its size is
  `len(tags) + 1`.
- Loss `sparse_categorical_crossentropy` (targets are integer ids, not one-hot), optimizer `adam`.
- About 5 epochs, `batch_size=32`, `validation_split=0.2`. More will overfit; the dataset is small.

In [ ]:
# ---------------- TASK 3 ----------------

# TODO 12: build `model_lstm` -- Embedding(mask_zero=True), LSTM(return_sequences=True),
#          TimeDistributed(Dense(..., activation='softmax'))

# TODO 13: compile it with the right loss and optimizer

# TODO 14: print model_lstm.summary()

# TODO 15: train it, keeping the result in `history_lstm`

# TODO 16: print the per-epoch table and the final epoch's four metrics

---
# Task 4: Evaluating the tagger correctly &nbsp;&nbsp;*(4 marks)*

> **Watch out — this is the trap in this assignment.** Most positions in `y_test` are **padding**,
> not real words. An accuracy computed over the whole array is meaningless: it is dominated by
> padding, and depending on how you compute it you can get a number that is far too high *or* far
> too low. Neither tells you how good your tagger is.
>
> **Report accuracy over real tokens only** — every position where the true tag id is not the
> padding id. Compute it yourself; do not simply quote whatever `model.evaluate()` prints.

### Part A — masked accuracy

Write `masked_accuracy(model, X, y)` returning the fraction of **non-padding** positions where the
predicted tag id equals the true one. Also print the accuracy `model_lstm.evaluate(...)` gives, and
**state in one line whether the two agree and why**.

### Part B — per-tag accuracy

Report accuracy separately for each of the 12 tags, sorted worst first. This shows *which* parts of
speech are hard, which is far more useful than a single number.

### Part C — tag new sentences

Write `tag_sentence(model, sentence)` taking a plain string and returning `(word, predicted_tag)`
pairs. Run it on:

* `The quick brown fox jumps over the lazy dog .`
* `He will book a flight to New York tomorrow .`
* `They are reading a book about the stock market .`

### Output your code must print

```
Masked accuracy (real tokens only): 0.____
Keras evaluate() accuracy:          0.____
[one line explaining whether they agree and why]

PER-TAG ACCURACY
tag      n      accuracy
...

The/DET quick/ADJ brown/ADJ fox/NOUN ...
```

### Hints

- `model.predict(X)` returns `(n, MAX_LEN, n_tags+1)`. Use `.argmax(-1)` for predicted ids.
- Build the mask as `y_true != 0` and compare only those positions.
- Words missing from `word_index` must map to the unknown id, exactly as in Task 2.
- Sentences 2 and 3 are chosen deliberately: **"book"** is a verb in one and a noun in the other.
  Check whether your tagger gets both right.
- Sentence 1 is ordinary English, but the Treebank is **Wall Street Journal text**. Expect some odd
  tags, and think about why.

In [ ]:
# ---------------- TASK 4 ----------------

def masked_accuracy(model, X, y):
    """
    Tagging accuracy over REAL tokens only, ignoring padded positions.

    Output: float
    """
    # TODO 17: predict, then take argmax over the tag axis
    # TODO 18: build a mask selecting positions where the TRUE tag is not padding
    # TODO 19: return the accuracy over masked positions only
    pass


def tag_sentence(model, sentence):
    """
    Tag a plain string.

    Input :  sentence (str)
    Output:  list of (word, predicted_tag) pairs
    """
    # TODO 20: split and lowercase the sentence, map words to ids (unknown -> the <unk> id)
    # TODO 21: pad to MAX_LEN the same way as Task 2, predict, take argmax
    # TODO 22: map predicted ids back to tag strings, pair them with the ORIGINAL words,
    #          and return only as many tags as there are words
    pass


# TODO 23: print the masked accuracy and model_lstm.evaluate()'s accuracy, plus one line
#          saying whether they agree and why

# TODO 24: print per-tag accuracy for all 12 tags, worst first

# TODO 25: tag the three sentences above and print them as word/TAG

---
# Task 5: Bidirectional LSTM and comparison &nbsp;&nbsp;*(4 marks)*

### The idea

Your LSTM reads strictly **left to right**. When it tags a word it has seen everything before it
and **nothing after it**. But the disambiguating words are often on the right:

> *He will **book** a flight.* &nbsp;&nbsp;vs&nbsp;&nbsp; *She read a **book** about markets.*

A **bidirectional** RNN runs two LSTMs — one forward, one backward — and joins their hidden states
at each position:

$$h_t = \left[\overrightarrow{h_t}\; ;\; \overleftarrow{h_t}\right]$$

So every word is tagged using the whole sentence. Wrapping a layer in `Bidirectional(...)` is all
Keras needs; note that it **doubles** the output width per step.

> Bidirectionality is only possible because the **entire** sentence is available up front. You
> could not do this in a language model that must predict the next word — which is why Lab 6 did
> not use it.

### What your code must create

| Name | Is |
|---|---|
| `model_bilstm` | Task 3's model with the LSTM wrapped in `Bidirectional` |
| `history_bilstm` | its training history |

Train it **exactly** as you trained the LSTM — same epochs, batch size and split — or the
comparison is not fair.

### Output your code must print

```
MODEL COMPARISON (masked accuracy on the test set)
  LSTM      params ______   accuracy 0.____
  BiLSTM    params ______   accuracy 0.____
  improvement: +_.__ percentage points

[the three sentences tagged by the BiLSTM]
[any words where the two models disagree]
```

### Then answer

**Q2.** How many percentage points did bidirectionality gain, and how many extra parameters did it
cost? Was it worth it?

**Q3.** List every word where the two models disagree. Pick one and explain, using right-hand
context, why seeing the rest of the sentence could change the decision. *(Look closely at the
proper noun in sentence 2.)* Also say whether **"book"** needed bidirectionality at all, or whether
left context was already enough.

**Q4.** From your per-tag table, which tags were hardest? Sentence 1 is plain English yet gets
several tags wrong — what does that tell you about the data the model was trained on?

### Hints

- Only one line of the model changes. Training, evaluation and `masked_accuracy` are reused as is.
- Call `keras.utils.set_random_seed(42)` again before building the second model, so the comparison
  is not distorted by different random initialisation.
- `model.count_params()` gives the parameter count.
- To find disagreements, tag the same sentence with both models and compare position by position.

In [ ]:
# ---------------- TASK 5 ----------------

# TODO 26: build `model_bilstm` -- identical to Task 3 but with the LSTM wrapped
#          in Bidirectional(...)

# TODO 27: compile and train it with EXACTLY the same settings, keeping `history_bilstm`

# TODO 28: print the comparison -- parameter count and masked accuracy for both models,
#          and the improvement in percentage points

# TODO 29: tag the three sentences from Task 4 with the BiLSTM and print them

# TODO 30: print any positions where the two models predict different tags

### Your answers

**Q1.** Where the LSTM's improvement was largest, and whether that matches the vanishing-gradient
explanation:

*Your answer:*

**Q2.** Gain from bidirectionality, and its cost in parameters:

*Your answer:*

**Q3.** Where the two models disagree, and what right-hand context explains it:

*Your answer:*

**Q4.** The hardest tags, and what sentence 1 reveals about the training data:

*Your answer:*

---
## Submission checklist

- [ ] `Runtime → Restart and run all` completes without errors
- [ ] **Task 1** — the three-way comparison table against the SimpleRNN baseline
- [ ] **Task 2** — statistics printed, and one example shown in both word and id form
- [ ] **Task 3** — `model_lstm.summary()` and the per-epoch table are visible
- [ ] **Task 4** — masked accuracy, the comparison with `evaluate()`, the per-tag table, and the
      three tagged sentences
- [ ] **Task 5** — both models compared on parameters and accuracy, BiLSTM tagging shown,
      disagreements listed
- [ ] All four questions answered
- [ ] Submit the `.ipynb` with all outputs saved

### Common mistakes

1. Changing something other than the recurrent layer in Task 1, making the comparison meaningless.
2. Using id 0 for a real word or a real tag — 0 must mean padding and nothing else.
3. Padding at the front (`padding='pre'`) for tagging, which misaligns words and tags.
4. Forgetting `return_sequences=True`, so the LSTM emits one vector instead of one per word.
5. Reporting raw accuracy over the whole padded array instead of over real tokens only.
6. Training the two taggers for different numbers of epochs.
7. Looking words up without lowercasing them, when the vocabulary was built lowercased.